# Teti et al. (2019) — DoubleML PLR Robustness
**Ashley Thompson — Capstone**

DoubleML Partially Linear Regression (Robinson 1988) robustness analysis for the Teti et al. (2019) replication. XGBoost nuisance models partial out nonlinear confounding so treatment coefficients have a causal interpretation under conditional unconfoundedness.

**Treatment channels (Teti):**
- `twitter_sent` — Bloomberg composite Twitter sentiment
- `twitter_neg_count` — raw negative tweet count (Teti SET 2/3 key predictor)
- `ts_b` — polarity-weighted index `(pos-neg)/(pos+neg+neu)`

**Outcome:** `pct_return = (px_close - px_open) / px_open × 100`

**Structure:**
1. Main ATE — current treatment → current return
2. Part 1 — lagged treatment → current return *(reverse causality check)*
3. No-reversal test — all `twitter_sent` lags simultaneously *(Gu & Kurov 2020 procedure)*
4. Part 2 — current treatment → future return *(impact persistence)*
5. Export LaTeX tables

> **Runtime → Change runtime type → T4 GPU** before running.

## 1. Install dependencies

In [ ]:
!pip install -q doubleml xgboost scikit-learn

## 2. Load data
Run Option A or Option B, then the GitHub setup cell.

In [ ]:
# Option A — Google Drive (data only)
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Capstone/panel_long.csv'

In [ ]:
# Option B — Manual upload (data only)
from google.colab import files
uploaded = files.upload()
DATA_PATH = 'panel_long.csv'

In [ ]:
# GitHub output setup — run after whichever data option above
from google.colab import userdata
import os, subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR     = '/content/Capstone'
OUTPUT_PATH  = REPO_DIR + '/output/'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone',
         f'https://{GITHUB_TOKEN}@github.com/ATHOMP-03/Capstone.git',
         REPO_DIR],
        check=True,
    )
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'Output path ready: {OUTPUT_PATH}')

## 3. Setup

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import doubleml as dml
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)
os.makedirs(OUTPUT_PATH, exist_ok=True)

long = pd.read_csv(DATA_PATH, parse_dates=['date'])
long = long.sort_values(['ticker', 'date']).reset_index(drop=True)
print(f'Loaded {len(long):,} rows x {long.shape[1]} cols')

HAS_POS_NEU = {'twitter_pos_count', 'twitter_neu_count'}.issubset(long.columns)
print(f'HAS_POS_NEU = {HAS_POS_NEU}  (ts_b exact formula available: {HAS_POS_NEU})')
long.head()

In [ ]:
# Variable construction
long['pct_return'] = (long['px_close'] - long['px_open']) / long['px_open'] * 100

if HAS_POS_NEU:
    denom = long['twitter_pos_count'] + long['twitter_neu_count'] + long['twitter_neg_count']
    long['ts_b'] = (long['twitter_pos_count'] - long['twitter_neg_count']) / denom.replace(0, np.nan)
else:
    long['ts_b'] = long['twitter_sent'].apply(
        lambda x: np.sign(x) * (x ** 2) if pd.notna(x) else np.nan
    )
    print('[INFO] ts_b using sign(s)*s^2 fallback (pos/neu counts unavailable)')

print(f"pct_return: mean={long['pct_return'].mean():.4f}  std={long['pct_return'].std():.4f}")
print(f"ts_b:       mean={long['ts_b'].mean():.4f}  std={long['ts_b'].std():.4f}")

In [ ]:
def make_xgb():
    return XGBRegressor(
        n_estimators     = 500,
        learning_rate    = 0.05,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        device           = 'cuda',
        tree_method      = 'hist',
        random_state     = 42,
        n_jobs           = -1,
    )

LAGS = [1, 2, 3, 5, 7]

# Excludes all three treatment channels; news_sent is a structural confounder
TETI_CONFOUNDERS = [
    'px_high', 'px_low', 'mkt_cap', 'total_equity', 'debt_to_equity',
    'volume', 'twitter_count', 'rsi_30', 'ma_50', 'news_sent',
]

TREATMENTS = [
    ('twitter_sent',      'Twitter sentiment (composite)'),
    ('twitter_neg_count', 'Negative tweet count'),
    ('ts_b',              'Polarity-weighted index (ts_b)'),
]

def run_plr(df, y_col, d_col, x_cols):
    data  = dml.DoubleMLData(
        df[[y_col, d_col] + x_cols].dropna().reset_index(drop=True),
        y_col=y_col, d_cols=d_col, x_cols=x_cols,
    )
    model = dml.DoubleMLPLR(data, ml_l=make_xgb(), ml_m=make_xgb(), n_folds=5, n_rep=20)
    model.fit()
    model.bootstrap(method='normal', n_rep_boot=1000)
    coef     = model.summary['coef'].iloc[0]
    pval     = model.summary['P>|t|'].iloc[0]
    ci_lower = model.confint(level=0.95)['2.5 %'].iloc[0]
    ci_upper = model.confint(level=0.95)['97.5 %'].iloc[0]
    return coef, pval, ci_lower, ci_upper

def run_plr_multi(df, y_col, d_cols, x_cols):
    keep  = [y_col] + d_cols + x_cols
    data  = dml.DoubleMLData(
        df[keep].dropna().reset_index(drop=True),
        y_col=y_col, d_cols=d_cols, x_cols=x_cols,
    )
    model = dml.DoubleMLPLR(data, ml_l=make_xgb(), ml_m=make_xgb(), n_folds=5, n_rep=20)
    model.fit()
    model.bootstrap(method='normal', n_rep_boot=1000)
    return model

print('Setup complete.')

## 4. Main ATE — current treatment → pct_return
Primary causal estimate. Each treatment estimated separately with XGBoost nuisance models controlling for all non-treatment confounders.

In [ ]:
main_rows = []

for col, desc in TREATMENTS:
    if col not in long.columns:
        print(f'[SKIP] {desc} — column not found')
        continue
    coef, pval, ci_lower, ci_upper = run_plr(long, 'pct_return', col, TETI_CONFOUNDERS)
    sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else ''
    print(f'{desc:<35} coef={coef:>9.6f}  p={pval:.4f}  CI=[{ci_lower:.6f}, {ci_upper:.6f}]  {sig}')
    main_rows.append({'label': desc, 'coef': coef, 'pval': pval, 'ci_lower': ci_lower, 'ci_upper': ci_upper})

main_df = pd.DataFrame(main_rows)
print('\nMAIN ATE TABLE')
print(main_df.to_string(index=False))

## 5. Part 1 — Lagged treatment → pct_return
*(Reverse causality check — univariate)*

Each lag estimated separately. If past sentiment predicts current returns, causality may run price → tweets rather than tweets → price.

**Pass:** only lag 1 borderline; lags 3-7 insignificant.

In [ ]:
# Build lagged treatment columns
for col, _ in TREATMENTS:
    if col not in long.columns:
        continue
    for lag_n in LAGS:
        long[f'{col}_lag{lag_n}'] = long.groupby('ticker')[col].shift(lag_n)

part1_rows = []

for lag_n in LAGS:
    print(f'\n--- Lag {lag_n} ---')
    for col, desc in TREATMENTS:
        if col not in long.columns:
            continue
        lag_col = f'{col}_lag{lag_n}'
        coef, pval, ci_lower, ci_upper = run_plr(long, 'pct_return', lag_col, TETI_CONFOUNDERS)
        sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else ''
        print(f'  lag{lag_n} | {desc:<30} coef={coef:>9.6f}  p={pval:.4f}  CI=[{ci_lower:.6f}, {ci_upper:.6f}]  {sig}')
        part1_rows.append({'label': desc, 'lag': lag_n, 'coef': coef, 'pval': pval, 'ci_lower': ci_lower, 'ci_upper': ci_upper})

part1_df = pd.DataFrame(part1_rows)
print('\nLAG DECAY TABLE (univariate)')
print(part1_df.sort_values(['lag', 'label']).to_string(index=False))

## 6. No-reversal test — all twitter_sent lags simultaneously
*(Gu & Kurov 2020 procedure)*

All lags estimated jointly so each coefficient is conditioned on the others. Correlated lags cannot absorb each other's variation.

**Pass:** lag 1 significant, lags 2–7 insignificant → permanent information effect  
**Fail:** lags 3–7 persist → momentum or reverse causality

In [ ]:
for col, desc in [('twitter_sent', 'Twitter sentiment'), ('ts_b', 'Polarity-weighted ts_b')]:
    if col not in long.columns:
        continue
    lag_cols = [f'{col}_lag{n}' for n in LAGS if f'{col}_lag{n}' in long.columns]

    nr_model   = run_plr_multi(long, 'pct_return', lag_cols, TETI_CONFOUNDERS)
    nr_summary = nr_model.summary
    nr_confint = nr_model.confint(level=0.95)

    print(f'\n--- {desc} ---')
    verdicts = {}
    for lag_col in lag_cols:
        coef     = nr_summary.loc[lag_col, 'coef']
        pval     = nr_summary.loc[lag_col, 'P>|t|']
        ci_lower = nr_confint.loc[lag_col, '2.5 %']
        ci_upper = nr_confint.loc[lag_col, '97.5 %']
        verdicts[lag_col] = pval < 0.05
        sig = '*' if pval < 0.05 else ''
        print(f'  {lag_col:<25} coef={coef:>9.6f}  p={pval:.4f}  CI=[{ci_lower:.6f}, {ci_upper:.6f}]  {sig}')

    lag1_col  = f'{col}_lag1'
    later_sig = [c for c in lag_cols if c != lag1_col and verdicts.get(c, False)]
    print(f'\n  VERDICT ({desc}):')
    if verdicts.get(lag1_col) and not later_sig:
        print('  PASS — lag 1 significant, lags 2-7 insignificant. Permanent information effect; no reverse causality.')
    elif not verdicts.get(lag1_col):
        print('  INCONCLUSIVE — lag 1 not significant.')
    else:
        print(f'  FAIL — significant persistence at: {", ".join(later_sig)}. Suggests momentum or reverse causality.')

## 7. Part 2 — Current treatment → pct_return_lead{n}
*(Impact persistence — research question)*

How long does today's tweet sentiment affect future returns?

**Hypothesis:** `twitter_sent` and `ts_b` effects decay by lead 2–3 (rapid tweet cycle absorption); `twitter_neg_count` may persist longer.

In [ ]:
# Create lead return variables
for lead_n in LAGS:
    long[f'pct_return_lead{lead_n}'] = long.groupby('ticker')['pct_return'].shift(-lead_n)

part2_rows = []

for lead_n in LAGS:
    outcome = f'pct_return_lead{lead_n}'
    print(f'\n--- Lead {lead_n} ---')
    for col, desc in TREATMENTS:
        if col not in long.columns:
            continue
        coef, pval, ci_lower, ci_upper = run_plr(long, outcome, col, TETI_CONFOUNDERS)
        sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else ''
        print(f'  lead{lead_n} | {desc:<30} coef={coef:>9.6f}  p={pval:.4f}  CI=[{ci_lower:.6f}, {ci_upper:.6f}]  {sig}')
        part2_rows.append({'label': desc, 'lead': lead_n, 'coef': coef, 'pval': pval, 'ci_lower': ci_lower, 'ci_upper': ci_upper})

part2_df = pd.DataFrame(part2_rows)
print('\nIMPACT PERSISTENCE TABLE')
print(part2_df.sort_values(['lead', 'label']).to_string(index=False))

## 8. Export — LaTeX tables

In [ ]:
def to_latex(rows, caption, label, period_col=''):
    def stars(p):
        if p < 0.01: return '^{***}'
        if p < 0.05: return '^{**}'
        if p < 0.10: return '^{*}'
        return ''

    period_hdr = f' & {period_col.title()}' if period_col else ''
    tab_spec   = r'{llcccc}' if period_col else r'{lcccc}'
    lines = [
        r'\begin{table}[htbp]', r'\centering',
        f'\\caption{{{caption}}}', f'\\label{{{label}}}',
        f'\\begin{{tabular}}{tab_spec}',
        r'\hline\hline',
        f'Treatment{period_hdr} & Coefficient & p-value & 95\\% CI Lower & 95\\% CI Upper \\\\',
        r'\hline',
    ]
    for r in rows:
        period_cell = f' & {int(r[period_col])}' if period_col else ''
        lbl = str(r['label']).replace('_', r'\_')
        lines.append(
            f"{lbl}{period_cell} & ${r['coef']:.6f}{stars(r['pval'])}$ "
            f"& {r['pval']:.4f} & {r['ci_lower']:.6f} & {r['ci_upper']:.6f} \\\\"
        )
    lines += [
        r'\hline',
        r'\multicolumn{5}{l}{\footnotesize DoubleML PLR, XGBoost (GPU), 5-fold, 20 reps, 1000 bootstrap draws.} \\',
        r'\multicolumn{5}{l}{\footnotesize $^{***}p<0.01$, $^{**}p<0.05$, $^{*}p<0.10$.} \\',
        r'\hline\hline', r'\end{tabular}', r'\end{table}',
    ]
    return '\n'.join(lines)


# Main ATE
with open(OUTPUT_PATH + 'teti_main_ate.tex', 'w') as f:
    f.write(to_latex(main_rows, 'Teti et al. (2019) --- DoubleML PLR Main ATE Estimates', 'tab:teti_main_ate'))
print('Saved teti_main_ate.tex')

# Lag decay (Part 1)
with open(OUTPUT_PATH + 'teti_lag_decay.tex', 'w') as f:
    f.write(to_latex(
        sorted(part1_rows, key=lambda r: (r['lag'], r['label'])),
        'Teti et al. --- Lagged Sentiment and Current Return (Reverse Causality Check)',
        'tab:teti_lag_decay', period_col='lag',
    ))
print('Saved teti_lag_decay.tex')

# Impact persistence (Part 2)
with open(OUTPUT_PATH + 'teti_impact_persistence.tex', 'w') as f:
    f.write(to_latex(
        sorted(part2_rows, key=lambda r: (r['lead'], r['label'])),
        'Teti et al. --- Current Sentiment and Future Return (Impact Persistence)',
        'tab:teti_impact_persistence', period_col='lead',
    ))
print('Saved teti_impact_persistence.tex')

In [ ]:
# Push outputs to GitHub
import os, subprocess

os.chdir(REPO_DIR)
subprocess.run(['git', 'config', 'user.email', 'ATHOMP-03@users.noreply.github.com'])
subprocess.run(['git', 'config', 'user.name',  'ATHOMP-03'])
subprocess.run(['git', 'add', 'output/'])

staged   = subprocess.run(['git', 'diff', '--cached', '--quiet'])
unpushed = subprocess.run(
    ['git', 'log', 'origin/main..HEAD', '--oneline'],
    capture_output=True, text=True
)

if staged.returncode != 0:
    subprocess.run(['git', 'commit', '-m', 'Add Teti DoubleML robustness results from Colab'], check=True)

if staged.returncode != 0 or unpushed.stdout.strip():
    subprocess.run(['git', 'push'], check=True)
    print('Pushed to GitHub.')
else:
    print('Nothing new to commit or push.')